# Projected Quantum Kernel — Full Model
Uses the pre-computed projections from IBM Heron R2 hardware.
Run cells top-to-bottom. No QPU access needed for Tracks 0/1/2.

## 1. Imports

In [2]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import f1_score, classification_report
from sklearn.metrics.pairwise import rbf_kernel
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 2. Load & Preprocess Data

In [3]:
# ── config ────────────────────────────────────────────────────────────────
DATA_DIR = 'data_tutorial/pqk'   # folder containing the 4 CSVs
MOTIFS   = ['motif', 'motif.1', 'motif.2', 'motif.3']
LABEL    = 'Nalm 6 Cytotoxicity'
THRESHOLD = 0.62    # above → toxic (1), below → non-toxic (-1)
# ──────────────────────────────────────────────────────────────────────────

import os

def load_and_preprocess(data_dir):
    tr = pd.read_csv(os.path.join(data_dir, 'train_data.csv'), encoding='unicode_escape')
    te = pd.read_csv(os.path.join(data_dir, 'test_data.csv'),  encoding='unicode_escape')

    for df in [tr, te]:
        df.replace(17, 14, inplace=True)
        df.columns = ['Cell Number','motif','motif.1','motif.2','motif.3','motif.4','Nalm 6 Cytotoxicity']

    def binarize(labels):
        lbl = labels.copy().astype(float)
        lbl[lbl >  THRESHOLD] =  1
        lbl[lbl <= THRESHOLD] = -1
        return lbl.astype(int).values

    train_labels = binarize(tr[LABEL])
    test_labels  = binarize(te[LABEL])

    # One-hot encode motifs
    tr_m = tr[MOTIFS].values
    te_m = te[MOTIFS].values

    min_c = min(tr_m.min(), te_m.min())
    tr_m -= min_c;  te_m -= min_c
    num_class = max(tr_m.max(), te_m.max()) + 1

    def one_hot(arr, nc):
        oh = np.eye(nc)[arr]
        return oh.reshape(arr.shape[0], -1)

    return (one_hot(tr_m, num_class),
            one_hot(te_m, num_class),
            train_labels, test_labels)

train_data, test_data, train_labels, test_labels = load_and_preprocess(DATA_DIR)

# Load pre-computed quantum projections
projections_train = pd.read_csv(os.path.join(DATA_DIR, 'projections_train.csv'), header=0).values
projections_test  = pd.read_csv(os.path.join(DATA_DIR, 'projections_test.csv'),  header=0).values

# Scale by π/2  (matches IBM tutorial)
projections_train = projections_train * (np.pi / 2)
projections_test  = projections_test  * (np.pi / 2)

print(f'train_data:        {train_data.shape}  labels: {np.bincount(train_labels+1)}')
print(f'projections_train: {projections_train.shape}')
print(f'test_data:         {test_data.shape}')
print(f'projections_test:  {projections_test.shape}')

FileNotFoundError: [Errno 2] No such file or directory: 'data_tutorial/pqk/train_data.csv'

## 3. Track 0 — Classical vs Quantum-Projected SVM Baseline

In [ ]:
PARAM_GRID = {
    'C':     [0.1, 0.5, 1, 2.5, 5, 7.5, 8.5, 10, 10.75, 15, 25, 50, 100],
    'gamma': ['auto','scale', 0.001, 0.005, 0.01, 0.02, 0.04, 0.05, 0.1, 0.25, 0.5, 1],
}
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)

def train_svm(X_train, y_train, X_test, y_test, label=''):
    gs = GridSearchCV(SVC(kernel='rbf'), PARAM_GRID, cv=cv,
                      scoring='f1_weighted', n_jobs=-1)
    gs.fit(X_train, y_train)
    pred = gs.predict(X_test)
    f1   = f1_score(y_test, pred, average='weighted')
    print(f'{label}: best={gs.best_params_}  F1={f1:.4f}')
    return gs, f1, pred

print('Training classical SVM...')
svm_c, f1_c, pred_c = train_svm(train_data,        train_labels,
                                 test_data,         test_labels, 'Classical')
print('Training quantum-projected SVM...')
svm_q, f1_q, pred_q = train_svm(projections_train, train_labels,
                                 projections_test,  test_labels, 'Quantum  ')

print(f'\nΔF1 (quantum − classical) = {f1_q - f1_c:+.4f}')

In [ ]:
# Bar chart
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(['Classical', 'Quantum-projected'], [f1_c, f1_q],
              color=['#3498db', '#e74c3c'], alpha=0.85, edgecolor='black')
for b, s in zip(bars, [f1_c, f1_q]):
    ax.text(b.get_x() + b.get_width()/2, s + 0.01, f'{s:.4f}',
            ha='center', fontweight='bold')
ax.set_ylabel('Weighted F1 (test)'); ax.set_ylim(0, 1)
ax.set_title(f'Track 0 baseline — ΔF1 = {f1_q - f1_c:+.4f}')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()

# Kernel heatmaps
def resolve_gamma(g, X):
    if g == 'scale': return 1.0 / (X.shape[1] * X.var())
    if g == 'auto':  return 1.0 / X.shape[1]
    return g

K_c = rbf_kernel(train_data,        gamma=resolve_gamma(svm_c.best_params_['gamma'], train_data))
K_q = rbf_kernel(projections_train, gamma=resolve_gamma(svm_q.best_params_['gamma'], projections_train))

idx = np.argsort(train_labels)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, K, title in zip(axes, [K_c[idx][:,idx], K_q[idx][:,idx]],
                               ['Classical kernel', 'Quantum-projected kernel']):
    im = ax.imshow(K, cmap='viridis', aspect='auto')
    ax.set_title(title); plt.colorbar(im, ax=ax)
plt.suptitle('Kernel matrices sorted by class label')
plt.tight_layout(); plt.show()

## 4. Track 1 — Anatomy of the Quantum Encoding
Which measurement basis (X / Y / Z) and which motif position drives the gain?

In [ ]:
# projections are laid out as  ⟨X⟩(60 cols) | ⟨Y⟩(60) | ⟨Z⟩(60)
bases = {
    'X': (0,  60),
    'Y': (60, 120),
    'Z': (120,180),
}

results_basis = {}
for name, (s, e) in bases.items():
    _, f1, _ = train_svm(projections_train[:, s:e], train_labels,
                         projections_test[:,  s:e], test_labels,
                         label=f'Basis {name}')
    results_basis[name] = f1

print()
print('Full quantum F1:', f1_q)
print('Classical F1:  ', f1_c)

In [ ]:
# Within each basis, compare the 4 motif positions (15 cols each)
results_pos = {}
for basis, (s, e) in bases.items():
    for pos in range(4):
        a, b = s + pos*15, s + pos*15 + 15
        key = f'{basis}_pos{pos+1}'
        _, f1, _ = train_svm(projections_train[:, a:b], train_labels,
                             projections_test[:,  a:b], test_labels, key)
        results_pos[key] = f1

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].bar(results_basis.keys(), results_basis.values(), color=['#e74c3c','#3498db','#2ecc71'])
axes[0].axhline(f1_c, ls='--', color='gray', label='Classical'); axes[0].legend()
axes[0].set_title('F1 by measurement basis'); axes[0].set_ylabel('Weighted F1')

axes[1].bar(results_pos.keys(), results_pos.values())
axes[1].axhline(f1_c, ls='--', color='gray', label='Classical'); axes[1].legend()
axes[1].set_title('F1 by basis × motif position'); axes[1].set_ylabel('Weighted F1')
plt.xticks(rotation=45); plt.tight_layout(); plt.show()

## 5. Track 2 — Data Scarcity & Robustness
Does the quantum advantage grow when training data is scarce?

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit

fractions = [0.1, 0.2, 0.35, 0.5, 0.7, 1.0]
lc_c, lc_q = [], []

for frac in fractions:
    if frac == 1.0:
        idx = np.arange(len(train_labels))
    else:
        sss = StratifiedShuffleSplit(n_splits=1, train_size=frac, random_state=RANDOM_STATE)
        idx, _ = next(sss.split(train_data, train_labels))

    _, f1c, _ = train_svm(train_data[idx],        train_labels[idx],
                          test_data,              test_labels, f'Classical  frac={frac}')
    _, f1q, _ = train_svm(projections_train[idx], train_labels[idx],
                          projections_test,       test_labels, f'Quantum     frac={frac}')
    lc_c.append(f1c); lc_q.append(f1q)

plt.figure(figsize=(7, 4))
n = [int(f * len(train_labels)) for f in fractions]
plt.plot(n, lc_c, 'o-', label='Classical',         color='#3498db')
plt.plot(n, lc_q, 's-', label='Quantum-projected', color='#e74c3c')
plt.fill_between(n, lc_c, lc_q, alpha=0.15, color='purple')
plt.xlabel('Training samples'); plt.ylabel('Weighted F1 (test)')
plt.title('Learning curves: quantum vs classical')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
# Noise robustness — add Gaussian noise to projections
noise_levels = [0.0, 0.05, 0.1, 0.2, 0.4, 0.8]
noise_f1_c, noise_f1_q = [], []

for sigma in noise_levels:
    noisy_tr_c = train_data        + np.random.normal(0, sigma, train_data.shape)
    noisy_te_c = test_data         + np.random.normal(0, sigma, test_data.shape)
    noisy_tr_q = projections_train + np.random.normal(0, sigma, projections_train.shape)
    noisy_te_q = projections_test  + np.random.normal(0, sigma, projections_test.shape)

    _, f1c, _ = train_svm(noisy_tr_c, train_labels, noisy_te_c, test_labels, f'C σ={sigma}')
    _, f1q, _ = train_svm(noisy_tr_q, train_labels, noisy_te_q, test_labels, f'Q σ={sigma}')
    noise_f1_c.append(f1c); noise_f1_q.append(f1q)

plt.figure(figsize=(7, 4))
plt.plot(noise_levels, noise_f1_c, 'o-', label='Classical',         color='#3498db')
plt.plot(noise_levels, noise_f1_q, 's-', label='Quantum-projected', color='#e74c3c')
plt.xlabel('Noise σ'); plt.ylabel('Weighted F1 (test)')
plt.title('Robustness to Gaussian noise')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 6. Bonus — Geometric Difference
Quantifies how different the quantum kernel is from the classical one.

In [ ]:
from scipy.linalg import inv, sqrtm

def geometric_difference(K1, K2):
    """g(K_c, K_q) per Huang et al. 2021."""
    n = K1.shape[0]
    K1_norm = K1 / np.trace(K1)
    K2_norm = K2 / np.trace(K2)
    sqrt_K2 = sqrtm(K2_norm)
    M = sqrt_K2 @ inv(K1_norm) @ sqrt_K2
    eigvals = np.linalg.eigvalsh(M)
    eigvals = np.clip(eigvals.real, 0, None)
    return float(np.sqrt(np.max(eigvals)))

g = geometric_difference(K_c, K_q)
print(f'Geometric difference g(K_classical, K_quantum) = {g:.4f}')
print('Higher g → quantum kernel captures genuinely different structure.')

NameError: name 'K_c' is not defined